In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import pickle
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
def rfefeature(indep_x,dep_y,n):
    rfelist=[]
    log_model=LogisticRegression(solver='lbfgs')
    svc=SVC(kernel = 'linear', random_state = 0)
    DT=DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
    RF=RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    rfemodellist=[log_model,svc,DT,RF]
    for i in rfemodellist:
        print(i)
        log_rfe=RFE(estimator=i, n_features_to_select=n)
        log_fit=log_rfe.fit(indep_x,dep_y)
        log_rfe_feature=log_fit.transform(indep_x)
        rfelist.append(log_rfe_feature)
    return rfelist
    

In [3]:
def split_scaler(indep_x,dep_y):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    x_train,x_test,y_train,y_test=train_test_split(indep_x,dep_y,test_size=0.25,random_state=42)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test

In [4]:
def cm_prediction(classifier,x_test):
    y_pred=classifier.predict(x_test)
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    from sklearn.metrics import classification_report
    report=classification_report(y_test,y_pred)
    from sklearn.metrics import accuracy_score
    Accuracy=accuracy_score(y_test,y_pred)
    return classifier,Accuracy,report,cm,x_test,y_test

In [5]:
def logistic(x_train,y_train,x_test):
    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state = 0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [6]:
def svc(x_train,y_train,x_test):
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    param_grid={'kernel':['rbf','linear','poly'],'gamma':['auto','scale']}
    classifier=GridSearchCV(SVC(probability=True),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [7]:
def Decision(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import GridSearchCV
    param_grid={'criterion':['gini','entropy','log_loss'],'splitter':['best','random']}
    classifier=GridSearchCV(DecisionTreeClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [8]:
def random(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GridSearchCV
    param_grid={'criterion':['gini','entropy','log_loss'],'max_features':['sqrt','log2'],'n_estimators':[100,200]}
    classifier=GridSearchCV(RandomForestClassifier(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,cm,x_test,y_test=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,cm,x_test,y_test

In [9]:
def rfeclassification(acclog,accsvc,accdt,accrf):
    rfe_dataframe=pd.DataFrame(index=['Logistic','svc','DT','RF'],columns=['Logistic','SVC','DecisionTree','RandomForest'])
    for number,idex in enumerate(rfe_dataframe.index):
        rfe_dataframe['Logistic'][idex]=acclog[number]
        rfe_dataframe['SVC'][idex]=accsvc[number]
        rfe_dataframe['DecisionTree'][idex]=accdt[number]
        rfe_dataframe['RandomForest'][idex]=accrf[number]
    return rfe_dataframe

In [10]:
dataset=pd.read_csv("preprocessed ILPD.csv")
df=dataset
df=pd.get_dummies(df,dtype=int,drop_first=True)
df

,Age,TB,DB,Alkphos,Sgpt,Sgot,TP,ALB,A/G Ratio,Selector,Gender_Male
0,65,0.7,0.10,187.00,16.0,18,6.8,3.3,0.90,1,0
1,62,5.3,2.95,481.75,64.0,100,7.5,3.2,0.74,1,1
2,62,5.3,2.95,481.75,60.0,68,7.0,3.3,0.89,1,1
3,58,1.0,0.40,182.00,14.0,20,6.8,3.4,1.00,1,1
4,72,3.9,2.00,195.00,27.0,59,7.3,2.4,0.40,1,1
...,...,...,...,...,...,...,...,...,...,...,...
578,60,0.5,0.10,481.75,20.0,34,5.9,1.6,0.37,0,1
579,40,0.6,0.10,98.00,35.0,31,6.0,3.2,1.10,1,1
580,52,0.8,0.20,245.00,48.0,49,6.4,3.2,1.00,1,1
581,31,1.3,0.50,184.00,29.0,32,6.8,3.4,1.00,1,1


In [11]:
indep_x=df.drop('Selector',axis=1)
dep_y=df['Selector']

In [12]:
indep_x.shape
dep_y.shape
print(indep_x.shape)
print(type(indep_x))


(583, 10)
<class 'pandas.core.frame.DataFrame'>


In [17]:
rfelist=rfefeature(indep_x,dep_y,5)
acclog=[]
accsvc=[]
accdt=[]
accrf=[]
for i in rfelist: 
    print(type(i))
    x_train, x_test, y_train, y_test=split_scaler(i,dep_y)   
    classifier,Accuracy,report,cm,x_test,y_test=logistic(x_train,y_train,x_test)
    acclog.append(Accuracy)
    classifier,Accuracy,report,cm,x_test,y_test=svc(x_train,y_train,x_test)
    accsvc.append(Accuracy)
    classifier,Accuracy,report,cm,x_test,y_test=Decision(x_train,y_train,x_test)
    accdt.append(Accuracy)
    classifier,Accuracy,report,cm,x_test,y_test=random(x_train,y_train,x_test)
    accrf.append(Accuracy)
result=rfeclassification(acclog,accsvc,accdt,accrf)

LogisticRegression()


C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

SVC(kernel='linear', random_state=0)
DecisionTreeClassifier(max_features='sqrt', random_state=0)
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
<class 'numpy.ndarray'>
Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
<class 'numpy.ndarray'>
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
<class 'numpy.ndarray'>
Fitting 5 folds for each of 6 candidates, totalling 30 fits


C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
<class 'numpy.ndarray'>
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\Sathish\AppData\Local\Temp\ipykernel_16648\4005863323.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  rfe_dataframe['Logistic'][idex]=acclog[number]
C:\Users\Sathish\AppData\Local\Temp\ipykernel_16648\4005863323.py:5: FutureWarn

In [18]:
result
#5

,Logistic,SVC,DecisionTree,RandomForest
Logistic,0.732877,0.746575,0.623288,0.705479
svc,0.705479,0.726027,0.671233,0.684932
DT,0.760274,0.746575,0.664384,0.767123
RF,0.767123,0.767123,0.657534,0.739726


In [14]:
result
#3

,Logistic,SVC,DecisionTree,RandomForest
Logistic,0.726027,0.746575,0.616438,0.678082
svc,0.739726,0.746575,0.657534,0.664384
DT,0.780822,0.746575,0.705479,0.780822
RF,0.780822,0.746575,0.719178,0.767123


In [16]:
result
#4

,Logistic,SVC,DecisionTree,RandomForest
Logistic,0.732877,0.746575,0.650685,0.705479
svc,0.732877,0.746575,0.589041,0.664384
DT,0.767123,0.760274,0.671233,0.753425
RF,0.767123,0.760274,0.691781,0.773973


In [19]:
def rfefeature(indep_x,dep_y,n):
    rfelist=[]
    log_model=LogisticRegression(solver='lbfgs')
    svc=SVC(kernel = 'linear', random_state = 0)
    DT=DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
    RF=RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    rfemodellist=[log_model,svc,DT,RF]
    for i in rfemodellist:
        print(i)
        log_rfe=RFE(estimator=i, n_features_to_select=n)
        log_fit=log_rfe.fit(indep_x,dep_y)
        log_rfe_feature=log_fit.transform(indep_x)
        mask=log_fit.get_support()
        selected_features=indep_x.columns[mask].tolist()
        print(selected_features)
        rfelist.append(log_rfe_feature)
    return selected_features

In [20]:
rfefeature(indep_x,dep_y,3)

LogisticRegression()


C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\envs\ai\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _che

['DB', 'TP', 'ALB']
SVC(kernel='linear', random_state=0)
['TP', 'ALB', 'A/G Ratio']
DecisionTreeClassifier(max_features='sqrt', random_state=0)
['Age', 'Alkphos', 'Sgot']
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
['Age', 'Alkphos', 'Sgot']


['Age', 'Alkphos', 'Sgot']

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
models = {"Logistic": LogisticRegression(),"svm": SVC(),"Decision": DecisionTreeClassifier(),"random": RandomForestClassifier()}
results = []
for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": acc
    })

results_df = pd.DataFrame(results)
print(results_df)

      Model  Accuracy
0  Logistic  0.767123
1       svm  0.767123
2  Decision  0.684932
3    random  0.719178
